Youtube Tutorial : https://www.youtube.com/@AritraSen
LangChain CrashCourse Playlist: https://youtube.com/playlist?list=PLOrU905yPYXItzOax1OUsgkehvlM7wIK5&si=xfzcB6SqtufSEFPc

In [1]:
! pip install langchain --q
! pip install chromadb --q
! pip install rank_bm25 --q
! pip install sentence_transformers lark --quiet # for creating embeddings

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cudf 23.8.0 requires cupy-cuda11x>=12.0.0, which is not installed.
apache-beam 2.46.0 requires dill<0.3.2,>=0.3.1.1, but you have dill 0.3.7 which is incompatible.
apache-beam 2.46.0 requires pyarrow<10.0.0,>=3.0.0, but you have pyarrow 11.0.0 which is incompatible.
cmudict 1.0.13 requires importlib-metadata<6.0.0,>=5.1.0, but you have importlib-metadata 6.0.1 which is incompatible.
dask-cuda 23.8.0 requires dask==2023.7.1, but you have dask 2023.9.0 which is incompatible.
dask-cuda 23.8.0 requires pandas<1.6.0dev0,>=1.3, but you have pandas 2.0.2 which is incompatible.
dask-cudf 23.8.0 requires dask==2023.7.1, but you have dask 2023.9.0 which is incompatible.
dask-cudf 23.8.0 requires pandas<1.6.0dev0,>=1.3, but you have pandas 2.0.2 which is incompatible.
distributed 2023.7.1 requires dask==2023.7.1, but yo

In [2]:
from langchain.retrievers import BM25Retriever, EnsembleRetriever
from langchain.vectorstores import Chroma

In [3]:
doc_list = [
    "I like apples",
    "I like oranges",
    "Apples and oranges are fruits",
    "I am still figuring out how to use Apple's macbook",
    "I don't have Apple's iPhone",
    "Macbook is very smooth and swift"
]

### BM25 Retriever keyword retriever

In [4]:
# initialize the bm25 retriever 
bm25_retriever = BM25Retriever.from_texts(doc_list)
bm25_retriever.k = 2

In [5]:
bm25_retriever.get_relevant_documents("Apple")

[Document(page_content='Macbook is very smooth and swift'),
 Document(page_content="I don't have Apple's iPhone")]

In [6]:
bm25_retriever.get_relevant_documents("orange")

[Document(page_content='Macbook is very smooth and swift'),
 Document(page_content="I don't have Apple's iPhone")]

### BGEEmbedding

In [7]:
from langchain.embeddings import HuggingFaceBgeEmbeddings

model_name = "BAAI/bge-base-en-v1.5"
model_kwargs = {'device': 'cuda'}
encode_kwargs = {'normalize_embeddings': True} # set True to compute cosine similarity
embeddings = HuggingFaceBgeEmbeddings(
    model_name=model_name,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs,
)

/opt/conda/lib/python3.10/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.23.5
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [8]:
from langchain.vectorstores import Chroma

# load embeddings into Chroma - need to pass docs , embedding function and path of the db

db = Chroma.from_texts(doc_list,
                       embedding=embeddings,
                       persist_directory='./db')

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [9]:
db_retriever = db.as_retriever(search_kwargs={"k": 2})

In [10]:
db_retriever.get_relevant_documents("orange")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[Document(page_content='I like oranges'),
 Document(page_content='Apples and oranges are fruits')]

### Ensemble Retriever

In [11]:
# initialize the ensemble retriever
ensemble_retriever = EnsembleRetriever(retrievers=[bm25_retriever, 
                                                   db_retriever],
                                       weights=[0.4, 0.6])

In [12]:
docs = ensemble_retriever.get_relevant_documents("How is macbook?")
docs

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[Document(page_content='Macbook is very smooth and swift'),
 Document(page_content="I am still figuring out how to use Apple's macbook"),
 Document(page_content="I don't have Apple's iPhone")]

In [13]:
docs = ensemble_retriever.get_relevant_documents("iPhone")
docs

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[Document(page_content="I don't have Apple's iPhone"),
 Document(page_content='I like apples'),
 Document(page_content='Macbook is very smooth and swift')]